In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file"
print("Environment OK.")

Environment OK.


In [11]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    # This is a placeholder tool -- it doesn't call a real weather API,
    # it just returns a fixed string so we can prove the whole pipeline works.
    return f"It's always sunny in {city}!"

agent = create_agent(model="openrouter:nvidia/nemotron-3.5-lightning:free",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result)

{'messages': [HumanMessage(content="What's the weather in San Francisco?", additional_kwargs={}, response_metadata={}, id='30210e99-c31e-4167-9c82-2eb8ffca841a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather for city "San Francisco". Use function get_weather.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to get weather for city "San Francisco". Use function get_weather.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3.5-lightning:free', 'id': 'gen-1788147731-q1bqnpg6qjlUBWGBr4FE', 'created': 1788147731, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a055e9-07d1-77e1-845c-f14c44b03e92-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 

# Initiate a model

In [12]:
from langchain.chat_models import init_chat_model
openrouter_model = init_chat_model('nvidia/nemotron-3.5-lightning:free',
                                   model_provider="openrouter")
response = openrouter_model.invoke("In one sentence, what is LangChain?")
print(response.content)

LangChain is an open-source framework for building context-aware applications by chaining large language model prompts with external tools, data sources, and agents.


In [16]:
print("text:             ", response.text)
print("content_blocks:   ", response.content_blocks)
print("id:               ", response.id)
print("tool_calls:       ", response.tool_calls)
print("content:         ",response.content)
print("metadata:        ",response.response_metadata)

text:              LangChain is an open-source framework for building context-aware applications by chaining large language model prompts with external tools, data sources, and agents.
content_blocks:    [{'type': 'reasoning', 'reasoning': 'Here\'s a thinking process:\n\n1.  **Analyze User Request:**\n   - User asks: "In one sentence, what is LangChain?"\n   - Constraint: Exactly one sentence (or at most one sentence, but typically "in one sentence" means a single concise sentence).\n\n2.  **Identify Key Concepts of LangChain:**\n   - LangChain is a framework/development framework for building applications powered by large language models (LLMs).\n   - It enables chaining/connecting LLMs with other data sources, tools, agents, and prompts.\n   - It supports modular components like prompts, chains, agents, memory, and tools.\n   - Goal: Simplify development of LLM-powered applications.\n\n3.  **Drafting - Attempt 1 (Mental):**\n   LangChain is a framework for developing applications tha

In [15]:
print(response.usage_metadata)

{'input_tokens': 25, 'output_tokens': 495, 'total_tokens': 520, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 511}}


# TripMate Agent


In [46]:
import requests
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
import json
from langchain.chat_models import init_chat_model
from pydantic import BaseModel,Field

In [10]:
load_dotenv()

True

## Weather Tool 

In [32]:
@tool
def get_weather(city: str) -> str:
    """
    Get the current weather information for a city.

    Args:
        city: Name of the city, for example "Vernon", "Toronto", or "Bangalore".

    Returns:
        Current temperature, feels-like temperature, weather condition,
        humidity, wind speed, and city name.
    """

    api_key = os.getenv("OPENWEATHER_API_KEY")

    if not api_key:
        return "Error: OPENWEATHER_API_KEY environment variable is not set."

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 404:
            return f"Could not find weather information for '{city}'."

        if response.status_code == 401:
            return "Error: Invalid OpenWeather API key."

        response.raise_for_status()

        data = response.json()

        weather = data["weather"][0]

        result = {
            "city": data["name"],
            "country": data["sys"]["country"],
            "temperature_celsius": data["main"]["temp"],
            "feels_like_celsius": data["main"]["feels_like"],
            "condition": weather["main"],
            "description": weather["description"],
            "humidity_percent": data["main"]["humidity"],
            "wind_speed_mps": data["wind"]["speed"]
        }

        return json.dumps(result)

    except requests.exceptions.Timeout:
        return "Weather API request timed out."

    except requests.exceptions.RequestException as e:
        return f"Weather API request failed: {str(e)}"

    except (KeyError, IndexError):
        return "Unexpected response received from the weather API."

In [8]:
get_weather.invoke({"city":"Bangalore"})

"{'city': 'Bengaluru', 'country': 'IN', 'temperature_celsius': 23.88, 'feels_like_celsius': 24.31, 'condition': 'Clouds', 'description': 'overcast clouds', 'humidity_percent': 76, 'wind_speed_mps': 3.09}"

### Creating Agent to use the tool

In [14]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", tools=[get_weather])

In [15]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "how are you?"}]})

#### in below response , there won't be any tool call beacuse question is not related to weather

In [16]:
print(result)

{'messages': [HumanMessage(content='how are you?', additional_kwargs={}, response_metadata={}, id='806513d5-dc10-4df5-a4aa-8ed65228af7b'), AIMessage(content="I'm doing great—thank you for asking. How can I help you today?", additional_kwargs={'reasoning_content': 'The user is asking how I am. I should respond politely. No need to use tools.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking how I am. I should respond politely. No need to use tools.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788811071-Hq0koTC4bGlck1DFPefL', 'created': 1788811071, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a07d72-ca5c-7e61-9e7e-70d808a61da5-0', tool_calls=[], invalid_tool_calls=

In [20]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "what is the temperature in Vernon in BC"}]})

In [21]:
print(result)

{'messages': [HumanMessage(content='what is the temperature in Vernon in BC', additional_kwargs={}, response_metadata={}, id='d111dda2-d084-4f1c-954a-51d43b00999b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the temperature in Vernon, BC. I need to call the get_weather function with city "Vernon". The function requires city name as a string. I\'ll call it.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for the temperature in Vernon, BC. I need to call the get_weather function with city "Vernon". The function requires city name as a string. I\'ll call it.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788811193-KpOAgdkg15Xfk4Lm7aS6', 'created': 1788811193, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0

## Let's explore weather in Gemma model

In [ ]:
gemma_model = init_chat_model(
    "google/gemma-3-12b-it",
    model_provider="openrouter",
)

In [35]:
@tool
def get_weather_gemma(city: str) -> str:
    """
    Get CURRENT weather information.

    MUST use this tool when the user asks for:
    - current weather
    - temperature
    - humidity
    - wind
    - weather condition
    - whether it is raining
    - weather in a specific city

    Do not answer current weather questions from your own knowledge.

    Args:
        city: The city to get current weather for.
    """
    return f"city {city} has summer"

In [36]:
get_weather_gemma.invoke('Bangalore')

'city Bangalore has summer'

In [43]:
gemma_model_tool = gemma_model.bind_tools([get_weather_gemma])

In [44]:
response = gemma_model_tool.invoke("tell me Vernon BC climate?")
print(response)

content='' additional_kwargs={} response_metadata={'model_name': 'google/gemma-3-12b-it', 'id': 'gen-1788816208-ARsTZs3Dy2j6BzaMleKI', 'created': 1788816208, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.96e-05, 'cost_details': {'upstream_inference_completions_cost': 2.25e-06, 'upstream_inference_prompt_cost': 1.735e-05, 'upstream_inference_cost': 1.96e-05}} id='lc_run--01a07dc1-310b-74f2-b10f-9075e8b89925-0' tool_calls=[{'name': 'get_weather_gemma', 'args': {'city': '{}'}, 'id': 'chatcmpl-tool-94e52eea982c66fd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 347, 'output_tokens': 15, 'total_tokens': 362, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


### tool_calls=[{'name': 'get_weather_gemma', 'args': {'city': '{}'}}], gemma unable to extract city name it always brings {}

#### so now add pydantic model with args_schema to get city name

In [52]:
class Weather_city(BaseModel):
    city: str = Field(
        description="The name of the city. Example: Bangalore, Toronto, or Vernon."
    )

In [53]:
@tool(args_schema=Weather_city)
def get_weather_gemma1(city: str) -> str:
    """
    Get CURRENT weather information.

    MUST use this tool when the user asks for:
    - current weather
    - temperature
    - humidity
    - wind
    - weather condition
    - whether it is raining
    - weather in a specific city

    Do not answer current weather questions from your own knowledge.

    Args:
        city: The city to get current weather for.
    """
    return f"city {city} has summer"

In [54]:
get_weather_gemma1.args

{'city': {'description': 'The name of the city. Example: Bangalore, Toronto, or Vernon.',
  'title': 'City',
  'type': 'string'}}

In [57]:
gemma_model_tool = gemma_model.bind_tools([Weather_city])
response = gemma_model_tool.invoke("tell me Vernon BC city climate?")
print(response)

content='' additional_kwargs={} response_metadata={'model_name': 'google/gemma-3-12b-it', 'id': 'gen-1788819239-BTRDM1zZeWkkdAL9f4dL', 'created': 1788819239, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.58e-05, 'cost_details': {'upstream_inference_completions_cost': 1.8e-06, 'upstream_inference_prompt_cost': 1.4e-05, 'upstream_inference_cost': 1.58e-05}} id='lc_run--01a07def-715a-7820-885a-142f09dace76-0' tool_calls=[{'name': 'Weather_city', 'args': {'city': '{}'}, 'id': 'chatcmpl-tool-8481f1654c36ce5d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 280, 'output_tokens': 12, 'total_tokens': 292, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


In [58]:
gemma_str_op = gemma_model.with_structured_output(get_weather_gemma1)
response = gemma_str_op.invoke("tell me Vernon BC city climate?")
print(response)

{'city': 'Vernon BC'}


In [56]:
gemma_model.profile

{'name': 'Gemma 3 12B',
 'release_date': '2025-03-13',
 'last_updated': '2025-03-13',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'tool_call_streaming': True}

# Pre -Built Tools 

##### This blog defines how to use the pre-defined tools here, for example we are giong to use Tavily search to extract restults from web

In [24]:
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
)
@tool('Tavily_Web_Search', description="Use this when the user wants to search the internet.")
def web_search(query:str) -> str:
    """Search the internet for current information."""
    return tavily_search_tool.invoke(query)

In [25]:
tripmate = create_agent(model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free", tools=[get_weather, web_search])

In [28]:
result = tripmate.invoke({"messages": [{"role": "user", "content": "who is Canada's Prime minister in 2020, use tool to search this"}]})

In [29]:
print(result)

{'messages': [HumanMessage(content="who is Canada's Prime minister in 2020, use tool to search this", additional_kwargs={}, response_metadata={}, id='e64e60f9-0020-4db2-aedb-cde4be51f942'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user is asking about Canada's Prime Minister in 2020. They want me to use a tool to search for this information. I can use the Tavily_Web_Search tool to find the answer. Let me search.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "The user is asking about Canada's Prime Minister in 2020. They want me to use a tool to search for this information. I can use the Tavily_Web_Search tool to find the answer. Let me search."}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1788812725-8IdSA4dvhiPR1Q4YKvse', 'created': 1788812725, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost